In [26]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json

## Example

In [ ]:
load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    default_headers={
        "HTTP-Referer": "https://localhost:8888", 
        "X-Title": "E-Commerce Portfolio App",
    }    
)

response = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    messages=[{"role": "user", "content": "Hello OpenRouter!"}]
)

print(response.choices[0].message.content)

Hello! How can I help you today?


## Create syllabus json

In [ ]:
markdown_base_path = "../../official_doc/exam_syllabus/machine-learning-engineer-associate-02-domain{domain_id}.md"

markdown_contents = []
for domain_id in range(1, 5):
    markdown_file_path = markdown_base_path.format(domain_id=domain_id)
    with open(markdown_file_path, "r") as md_file:
        md_content = md_file.read()
        markdown_contents.append(md_content)

json_format = {
  "domain_id": "string",
  "domain_name": "string",
  "task_id": "string",
  "task_name": "string",
  "skill_id": "string",
  "skill_name": "string"
}

json_example = {
  "domain_id": "1",
  "domain_name": "Data Engineering",
  "task_id": "1",
  "task_name": "Data Collection",
  "skill_id": "1",
  "skill_name": "Extract data from data sources."
}

output = []
for markdown_content in markdown_contents:
    json_prompt = f"""
    You are provided with a markdown file containing the a content domain of AWS MLA-C02 Exam.
    Based on a content, create a json file with the following structure:
    {json_format}

    For example a skill item: + Skill 1.1.1: Extract data from data sources.
    The corresponding json object would be:

    {json_example}

    Restriction: Do not include any additional information or explanations in the output. Only provide the JSON representation of the skills extracted from the markdown content.
    
    Here is the markdown file:
    {markdown_content}
    """
    
    response = client.chat.completions.create(
        model="z-ai/glm-5.3",
        messages=[{"role": "user", "content": json_prompt}],
        response_format={"type": "json_object"}
    )

    response_content = response.choices[0].message.content
    output.append(response_content)
    


In [32]:
final_output = []
for item in output:
    items = json.loads(item).get("skills", [])
    final_output += items
with open("aws_mla_c02_skills.json", "w") as json_file:
    json.dump(final_output, json_file, indent=4)

## Notes creation

In [5]:
note_system_prompt = """
You are an AWS Examination expert and you are responsible in providing accurate and detailed answers to questions related to AWS services, architecture, and best practices. 
Your responses should be clear, concise, and informative, catering to both beginners and experienced professionals in the field of cloud computing.
"""

note_prompt_1 = """
You are now responsible in create study notes for the AWS Certified Machine Learning - Specialty (MLS-C02) exam."""
